In [1]:
BUCKET = "crypto-bigdata-2026"

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"s3://{BUCKET}/trusted/crypto_prices/")

print(f"Total filas: {df.count():,}")
df.printSchema()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
3,application_1779309328797_0005,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total filas: 6,507
root
 |-- SNo: integer (nullable = true)
 |-- Symbol: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: double (nullable = true)
 |-- Marketcap: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- daily_return: double (nullable = true)
 |-- volatility: double (nullable = true)
 |-- Name: string (nullable = true)

In [2]:
from pyspark.sql.functions import col, avg, stddev, min, max, round as r, count

print("=== Estadísticas descriptivas por moneda ===")
df.groupBy("Name") \
  .agg(
    count("Close").alias("dias"),
    r(avg("Close"), 2).alias("precio_promedio"),
    r(stddev("Close"), 2).alias("precio_stddev"),
    r(min("Close"), 2).alias("precio_min"),
    r(max("Close"), 2).alias("precio_max"),
    r(avg("Volume"), 0).alias("volumen_promedio"),
    r(avg("volatility"), 3).alias("volatilidad_prom")
  ) \
  .orderBy("precio_promedio", ascending=False) \
  .show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== Estad?sticas descriptivas por moneda ===
+---------+----+---------------+-------------+----------+----------+----------------+----------------+
|     Name|dias|precio_promedio|precio_stddev|precio_min|precio_max|volumen_promedio|volatilidad_prom|
+---------+----+---------------+-------------+----------+----------+----------------+----------------+
|  Bitcoin|1648|       11838.47|     13157.36|    777.76|  63503.46| 1.9761967197E10|           5.304|
| Ethereum|1648|         500.93|       644.82|      8.17|    4168.7|   9.245476273E9|           7.054|
|   Solana| 452|          10.47|        14.11|      0.52|     55.91|    1.95675061E8|          13.676|
|Chainlink|1385|           6.31|          9.9|      0.13|      52.2|    6.92360796E8|           11.08|
|  Cardano|1374|           0.26|         0.41|      0.02|      2.31|    8.93418323E8|            9.05|
+---------+----+---------------+-------------+----------+----------+----------------+----------------+

In [3]:
print("=== P3: Correlacion Volumen vs Precio de cierre ===")
for coin in ['Bitcoin', 'Ethereum', 'Solana', 'Cardano', 'Chainlink']:
    df_c = df.filter(col("Name") == coin)
    corr = df_c.stat.corr("Volume", "Close")
    interpretacion = "fuerte" if abs(corr) > 0.7 else "moderada" if abs(corr) > 0.4 else "debil"
    print(f"  {coin:10s}: r = {corr:.4f} ({interpretacion})")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== P3: Correlacion Volumen vs Precio de cierre ===
  Bitcoin   : r = 0.7396 (fuerte)
  Ethereum  : r = 0.7352 (fuerte)
  Solana    : r = 0.8061 (fuerte)
  Cardano   : r = 0.7565 (fuerte)
  Chainlink : r = 0.2238 (debil)

In [ ]:
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt

vol_pd = df.groupBy("Name") \
    .agg(r(avg("volatility"), 3).alias("volatilidad_pct")) \
    .orderBy("volatilidad_pct", ascending=False) \
    .toPandas()

colors = ['#f7931a','#627eea','#14f195','#4a90d9','#375bd2']
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(vol_pd['Name'], vol_pd['volatilidad_pct'], color=colors)
ax.set_xlabel('Volatilidad promedio (%)')
ax.set_title('P2: Volatilidad historica por criptomoneda')
for bar, val in zip(bars, vol_pd['volatilidad_pct']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('/tmp/volatilidad.png', dpi=100, bbox_inches='tight')
plt.show()
print("Grafico guardado")

In [4]:
import subprocess
subprocess.run(['pip', 'install', 'numpy', 'matplotlib'], capture_output=True)
print("Instalado")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Instalado

In [ ]:
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, avg, round as r

vol_pd = df.groupBy("Name") \
    .agg(r(avg("volatility"), 3).alias("volatilidad_pct")) \
    .orderBy("volatilidad_pct", ascending=False) \
    .toPandas()

colors = ['#f7931a','#627eea','#14f195','#4a90d9','#375bd2']
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(vol_pd['Name'], vol_pd['volatilidad_pct'], color=colors)
ax.set_xlabel('Volatilidad promedio (%)')
ax.set_title('P2: Volatilidad historica por criptomoneda')
for bar, val in zip(bars, vol_pd['volatilidad_pct']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('/tmp/volatilidad.png', dpi=100, bbox_inches='tight')
plt.show()
print("Grafico guardado")

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'numpy', 'matplotlib'],
    capture_output=True, text=True
)
print(result.stdout[-500:])
print(result.stderr[-500:])

In [ ]:
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, avg, round as r

vol_pd = df.groupBy("Name") \
    .agg(r(avg("volatility"), 3).alias("volatilidad_pct")) \
    .orderBy("volatilidad_pct", ascending=False) \
    .toPandas()

colors = ['#f7931a','#627eea','#14f195','#4a90d9','#375bd2']
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(vol_pd['Name'], vol_pd['volatilidad_pct'], color=colors)
ax.set_xlabel('Volatilidad promedio (%)')
ax.set_title('P2: Volatilidad historica por criptomoneda')
for bar, val in zip(bars, vol_pd['volatilidad_pct']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('/tmp/volatilidad.png', dpi=100, bbox_inches='tight')
plt.show()
print("Grafico guardado")

In [ ]:
sc.install_pypi_package("numpy")
sc.install_pypi_package("matplotlib")

In [5]:
from pyspark.sql.functions import avg, round as r, col

# P2: Volatilidad - calcular en Spark, graficar en pandas
vol_pd = df.groupBy("Name") \
    .agg(r(avg("volatility"), 3).alias("volatilidad_pct")) \
    .orderBy("volatilidad_pct", ascending=False) \
    .toPandas()

print("=== P2: Volatilidad por moneda ===")
print(vol_pd.to_string(index=False))

# P5: Rendimiento por mes
rend_pd = df.groupBy("Month") \
    .agg(r(avg("daily_return"), 4).alias("rendimiento_pct")) \
    .orderBy("Month") \
    .toPandas()

print("\n=== P5: Rendimiento promedio por mes ===")
print(rend_pd.to_string(index=False))

# P4: Precio mensual BTC
btc_pd = df.filter(col("Name") == "Bitcoin") \
    .filter(col("Year").between(2020, 2024)) \
    .groupBy("Year", "Month") \
    .agg(r(avg("Close"), 2).alias("precio_promedio")) \
    .orderBy("Year", "Month") \
    .toPandas()

print("\n=== P4: Precio mensual Bitcoin 2020-2024 ===")
print(btc_pd.to_string(index=False))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
Pandas >= 1.0.5 must be installed; however, it was not found.
Traceback (most recent call last):
  File "/mnt1/yarn/usercache/livy/appcache/application_1779309328797_0005/container_1779309328797_0005_01_000001/pyspark.zip/pyspark/sql/pandas/conversion.py", line 86, in toPandas
    require_minimum_pandas_version()
  File "/mnt1/yarn/usercache/livy/appcache/application_1779309328797_0005/container_1779309328797_0005_01_000001/pyspark.zip/pyspark/sql/pandas/utils.py", line 34, in require_minimum_pandas_version
    raise ImportError(
ImportError: Pandas >= 1.0.5 must be installed; however, it was not found.



In [6]:
from pyspark.sql.functions import avg, round as r, col

print("=== P2: Volatilidad por moneda ===")
df.groupBy("Name") \
    .agg(r(avg("volatility"), 3).alias("volatilidad_pct")) \
    .orderBy("volatilidad_pct", ascending=False) \
    .show()

print("=== P4: Precio mensual Bitcoin 2020-2024 ===")
df.filter(col("Name") == "Bitcoin") \
    .filter(col("Year").between(2020, 2024)) \
    .groupBy("Year", "Month") \
    .agg(r(avg("Close"), 2).alias("precio_promedio")) \
    .orderBy("Year", "Month") \
    .show(60)

print("=== P5: Rendimiento promedio por mes ===")
df.groupBy("Month") \
    .agg(r(avg("daily_return"), 4).alias("rendimiento_pct")) \
    .orderBy("Month") \
    .show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== P2: Volatilidad por moneda ===
+---------+---------------+
|     Name|volatilidad_pct|
+---------+---------------+
|   Solana|         13.676|
|Chainlink|          11.08|
|  Cardano|           9.05|
| Ethereum|          7.054|
|  Bitcoin|          5.304|
+---------+---------------+

=== P4: Precio mensual Bitcoin 2020-2024 ===
+----+-----+---------------+
|Year|Month|precio_promedio|
+----+-----+---------------+
|2020|    1|        8389.27|
|2020|    2|        9630.72|
|2020|    3|        6871.02|
|2020|    4|        7224.48|
|2020|    5|        9263.15|
|2020|    6|        9489.23|
|2020|    7|         9589.9|
|2020|    8|       11652.39|
|2020|    9|       10660.28|
|2020|   10|       11886.98|
|2020|   11|       16645.76|
|2020|   12|       21983.14|
|2021|    1|       34761.65|
|2021|    2|        46306.8|
|2021|    3|       54998.01|
|2021|    4|       57206.72|
|2021|    5|       46443.29|
|2021|    6|       35845.15|
|2021|    7|       34234.45|
+----+-----+---------------+


In [7]:
print("=== Estadisticas descriptivas completas ===")
df.groupBy("Name") \
    .agg(
        r(avg("Close"), 2).alias("precio_promedio"),
        r(avg("Volume"), 0).alias("volumen_promedio"),
        r(avg("volatility"), 3).alias("volatilidad_prom"),
        r(avg("daily_return"), 4).alias("retorno_diario_prom")
    ) \
    .orderBy("precio_promedio", ascending=False) \
    .show()

print("=== Correlacion Volumen vs Precio ===")
for coin in ['Bitcoin','Ethereum','Solana','Cardano','Chainlink']:
    corr = df.filter(col("Name")==coin).stat.corr("Volume","Close")
    print(f"  {coin:10s}: r = {corr:.4f}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== Estadisticas descriptivas completas ===
+---------+---------------+----------------+----------------+-------------------+
|     Name|precio_promedio|volumen_promedio|volatilidad_prom|retorno_diario_prom|
+---------+---------------+----------------+----------------+-------------------+
|  Bitcoin|       11838.47| 1.9761967197E10|           5.304|             0.3035|
| Ethereum|         500.93|   9.245476273E9|           7.054|             0.4928|
|   Solana|          10.47|    1.95675061E8|          13.676|             1.2259|
|Chainlink|           6.31|    6.92360796E8|           11.08|             0.6278|
|  Cardano|           0.26|    8.93418323E8|            9.05|             0.5971|
+---------+---------------+----------------+----------------+-------------------+

=== Correlacion Volumen vs Precio ===
  Bitcoin   : r = 0.7396
  Ethereum  : r = 0.7352
  Solana    : r = 0.8061
  Cardano   : r = 0.7565
  Chainlink : r = 0.2238